# Sequential pricing: a three-month renewal price policy

A price changes later engagement and the prices that the pricing engine sets next, and engagement
steers those prices. This notebook estimates the effect of a price policy on month-3 renewal and
revenue with longitudinal TMLE. The policy modifies the prices that the engine would set, and it
stays inside the price band that the data support. Each step shows its code, its output, and what
the output tells you. [Modified treatment policies at a
node](../technical-reference/longitudinal-tmle.md#modified-treatment-policies-at-a-node) gives the
estimator.

## The situation

A subscription retailer sells a monthly plan. In each of three months, a pricing engine sets the
renewal price of each subscriber. Before each price, the engine scores whether the subscriber is
engaged that month. The engine reads the usage, the engagement, and the gap between its last price
and that month's centre. It draws a price inside a band whose limits rise with engagement. After
the third price, the subscriber renews or lapses.

This page uses a synthetic law. [The examples index](index.md#the-retailer-program) states how it
differs from the offers law of the retailer program.

| element | on this page |
| --- | --- |
| data | a wide frame of 10000 synthetic subscribers, one row each, with no missing value |
| time order | `usage` is recorded once. In months 1 to 3, the engine scores engagement and then sets the price. After the third price, the subscriber renews or lapses |
| treatment | the renewal price of each month, inside the band $[4 + 2e, 8 + 2e]$ for the month's engagement $e$. The centre of the band is $6 + 2e$ |
| outcome | `renewed`, the renewal at the month-3 price. `revenue` is `renewed` times the month-3 price |
| observed | the usage, three engagement scores, the three prices that the engine set, and the renewal |
| counterfactual | the renewal and the revenue under a price policy that differs from the prices the engine set. The frame never shows them |

### The causal graph

The graph draws one month. The pattern of month $t$ repeats for $t = 1, 2, 3$.

| node | meaning |
| --- | --- |
| $W$ | the usage score |
| $L_t$ | the engagement score of month $t$, recorded before the price |
| $A_t$ | the renewal price of month $t$ |
| $Y$ | the renewal at the month-3 price |
| $R$ | the month-3 revenue, $Y A_3$ |
| $U$ | an unrecorded cause |

```text
 W (usage)  -->  L_t, A_t, and Y                       for t = 1, 2, 3

 One month t, with L_0 and A_0 at their starting values:

   L_t      -->  A_t          the band and its centre move with engagement
   L_{t-1}  -->  A_t          the engine reads the last gap from the last centre
   A_{t-1}  -->  A_t          the engine carries part of the last price gap
   A_{t-1}  -->  L_t          a price above the centre lowers the next engagement
   L_{t-1}  -->  L_t          engagement persists
   L_t      -->  Y            for t = 1, 2, 3
   A_t      -->  Y            for t = 1, 2, 3
   Y        -->  R            R = Y A_3

 U (unrecorded cause)  - - ->  A_t and Y              (assumed absent)
```

| arrow or absent arrow | what it asserts | role | where the law sets it |
| --- | --- | --- | --- |
| `W` to `L_t`, `A_t`, and `Y` | usage raises engagement (`+0.6` on the logit), the price mean (`+0.3`), and the renewal (`+0.3` on the logit) | baseline confounder | `engaged_usage`, `price_usage`, `renew_usage` |
| `L_t` to `A_t` | the centre and both band limits rise by `2` with engagement | confounder of the price. It also sets the support | `centre_engaged`, `band_below`, `band_above` |
| `A_{t-1}` and `L_{t-1}` to `A_t` | the engine adds `0.3` of the last price gap to the price mean. The gap reads the last centre, so `L_{t-1}` enters the price mean with `-0.6` | engine persistence. A policy must carry it | `price_gap`, `centre_engaged` |
| `A_{t-1}` and `L_{t-1}` to `L_t` | a price above the centre lowers engagement (`-0.35` on the gap). Last month's engagement raises it by `1.5` (`0.8` direct and `0.7` through the centre of the gap) | the treatment changes a later confounder | `engaged_gap`, `engaged_lag` |
| `A_3`, and `A_1` and `A_2`, to `Y` | the month-3 gap lowers the renewal (`-0.45` on the logit). Each earlier gap lowers it by `0.1`, as a reference price | the effect of the price | `renew_gap`, `renew_lag` |
| `L_3`, and `L_1` and `L_2`, to `Y` | month-3 engagement raises the renewal by `1.8` (`0.9` direct and `0.9` through the centre of the gap). Each earlier engagement raises it by `0.2` through the centre of its lag | a path from the confounder to the outcome | `renew_engaged`, `renew_lag`, `centre_engaged` |
| no `U` with arrows into `A_t` and `Y` | the engine logs every input it reads, so no unrecorded cause moves both the price and the renewal | encoded, and open. This is strong sequential randomization, and the engine log is the argument for it. It is scientific judgment | none, an assumption |
| no arrow between customers | one customer's price does not change another customer's renewal | encoded. This is no interference, and a field of the protocol | none, an assumption |

**Reading the graph.** Two paths run from a price to the renewal through a later variable. One runs
through the next engagement: $A_t \to L_{t+1} \to Y$. The other runs through the engine's next
price: $A_t \to A_{t+1} \to Y$. Engagement $L_{t+1}$ also causes $A_{t+1}$. 

One outcome regression that holds the later engagement and the later prices at their recorded values
blocks both paths. A regression that omits engagement leaves every price confounded. Step 8 shows
both failures.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| state a price policy as a modified treatment policy that stays inside the band | the next section and Step 5 |
| read from the graph why one regression fails and the sequential g-formula is needed | the graph and Step 8 |
| read the label `supported` and say what it covers | Steps 6 and 7 and the method evidence |
| read the support of a larger price move, and say why the effect saturates | Steps 9 and 10 |

## The question, the estimand, and the method

The pricing lead asks one question. Suppose the firm had raised each subscriber's monthly renewal
price by 12% for three months. The raise applies wherever the raised price stays inside that month's
price band, and the engine's price stays elsewhere. What share would renew in month 3? What would
month-3 revenue be?

Both answers compare with the prices that the engine set. A second policy cuts each price by 15%
wherever the cut price stays inside the band, and otherwise keeps the engine's price.

| part of the question | on this page |
| --- | --- |
| target population | active monthly subscribers in a cancellation-free window, as the protocol of Step 4 states |
| intervention | policy $d$, one map of the engine's price for each of months 1 to 3. `price_up` multiplies the price by `1.12` where the raised price stays below the month's ceiling $u_t$, and keeps the engine's price elsewhere. `price_down` multiplies it by 0.85 where the cut price stays above the month's floor $l_t$, and keeps the engine's price elsewhere. Each policy reads the month's engagement, which is recorded before the price. `natural_prices` is the engine's own prices |
| counterfactual outcome | $Y^{d}$ and $R^{d}$, the renewal and the revenue if each month's price had been the policy's price, and the later engagement and the engine's later prices had responded |
| estimand | $\psi_d = E[Y^{d}]$, the renewal share under $d$, and $E[R^{d}]$, the mean revenue |
| contrasts | $\psi_d - \psi_{\mathrm{natural}}$ for `price_up` and `price_down`, for the renewal and for the revenue |

The policy keeps the engine's price at the band edge, for a reason. A pure 12% scale assigns some
subscribers a price above the ceiling. The fit refuses that plan before any learner, because no
subscriber is observed at that price.

A rule that raised the price and capped it at the ceiling would put a point mass at the ceiling. Its
map is not invertible there, and the theory needs an invertible map ([Díaz, Williams, Hoffman and Schenck 2023](https://doi.org/10.1080/01621459.2021.1955691),
Assumption 4). The plans here are piecewise smooth and invertible. The held share is the share of
subscribers whose price a plan leaves unchanged.

A fixed price for every subscriber answers a different question. The fit refuses a static price on a
continuous node, because no regular estimator of its mean exists. An elasticity regression of the
renewal on the price does not answer the question either. Step 8 shows why.

| assumption | what it means here | checkable from data, or scientific judgment |
| --- | --- | --- |
| strong sequential randomization | at every month, each common cause of the price, the later engagement and prices, and the renewal is in the recorded history | scientific judgment. The argument is the engine log, which records every input the engine reads. No data test exists |
| positivity | the price that a plan assigns lies in the support of the engine's price given the history | the declared band holds by design. The density ratios of Step 9 show the empirical part |
| consistency | a price is the quoted renewal price, unrounded, and a plan quotes the new price for one month | scientific judgment |
| no interference | one subscriber's price does not change another subscriber's renewal | scientific judgment. One account belongs to one subscriber |

Each method feature answers one feature of the problem.

| feature of the problem | feature of the method |
| --- | --- |
| an earlier price changes engagement and the engine's next price, and engagement confounds the next price | sequential regression, which evaluates the g-formula with one regression for each month |
| the price is continuous, and the policy moves it | a modified treatment policy at a continuous node. The weight is a density ratio of the plan's price law to the engine's price law. `continuous_treatment=` declares the band |
| the engine's price law is unknown | `ratio="classifier"`, which estimates each density ratio by classification. The route is the one the registered study of this law uses |
| the nuisance models are main-terms or degree-2 parametric models | no cross-fitting. The registered cells for this configuration fit in sample |
| revenue is continuous | `q_bounds`, the declared support of the revenue |
| the question compares policies with the engine's prices | `RegimeContrast` against `natural_prices` |

Four estimators could answer the question. The table states what each one reads and when to use it.
[Modified treatment policies at a
node](../technical-reference/longitudinal-tmle.md#modified-treatment-policies-at-a-node) and [the
interval under heavy cumulative density
ratios](../technical-reference/longitudinal-tmle.md#the-interval-under-heavy-cumulative-density-ratios)
give the conditions.

| estimator | what it reads | when to use it |
| --- | --- | --- |
| LTMLE | the outcome regressions and the density ratios, with a targeting step | the default of this page. It is consistent when every outcome regression is correct, or when every density ratio is correct |
| SDR | the same, with no targeting step | when the ratio or the outcome regression could be wrong at different months, because it needs only one of the two at each month. Its interval runs short of nominal under heavy cumulative density ratios, and `price_down` has heavy ones |
| g-computation | the outcome regressions only | when every outcome regression is correct. It adds no protection against a wrong regression |
| IPW | the density ratios only | never for this plan. `IPWMethod` refuses a modified treatment policy, because `SequentialIPW` estimates no density ratio for it |

The table defines the terms this page uses most.

| term | plain meaning |
| --- | --- |
| node | one decision time. Here, the price of month 1, 2, or 3 |
| history | every column recorded before the price at a node |
| natural price | the price the engine sets. The code calls the engine's own prices `natural_prices` |
| [modified treatment policy](../technical-reference/point-treatment-tmle.md#modified-treatment-policies) | a rule that maps the price the engine would set to a new price. It reads the history |
| price band and support | the interval of prices the engine can set given the history, here $[4 + 2e, 8 + 2e]$. The analyst declares it |
| held share | the share of subscribers whose price a plan leaves unchanged because the moved price would leave the band |
| density ratio | the density of the plan's price divided by the density of the engine's price, at the engine's price. It weights each subscriber |
| effective n | the sample size that equal weights would give with the same variance. It falls as the weights grow uneven |
| [nuisance](../technical-reference/point-treatment-tmle.md) | a model the estimate needs but the question does not ask about. Here, the outcome, pseudo-outcome, and density-ratio models |
| [targeting](../user-guide/methods-learners.md#targeting-and-bounds) | a small update to each node's regression, weighted by the density ratios, that removes first-order bias |
| [influence curve](../technical-reference/inference.md) | how much each subscriber moves the estimate. Its variance gives the standard error |

## Step 1: set up

The setup imports the libraries and prints the installed `cleverly` version. Every fit below passes
its learners, its cross-fitting setting, and its random seed explicitly, so a rerun reproduces the
stored outputs.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.4


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.

## Step 2: the data

The generator `make_sequential_pricing` returns a wide frame with one row per subscriber. The code
prints the shape and the column names. It counts missing values, and it prints the true values of
the synthetic law.

The seed follows a fixed rule. The page uses 1000, the first seed of the range 1000 to 1199 that the
review probe of Step 6 sweeps. The rule was set before the page ran with this seed. No draw was
chosen for its results, so an interval on this page can miss its true value.

In [2]:
from cleverly.datasets import make_sequential_pricing

frame, truth = make_sequential_pricing(10_000, seed=1000)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head(6).round(3))
print()
print("missing values:", int(frame.isna().sum().sum()))
print("known values of the synthetic law:")
for plan in ("natural_prices", "price_up", "price_down"):
    renewal = truth[f"ey_regimen[{plan}]"]
    revenue = truth[f"ey_revenue[{plan}]"]
    print(f"  {plan:15s} mean renewal {renewal:.4f}  mean revenue {revenue:.4f}")
for plan in ("price_up", "price_down"):
    renewal = truth[f"ate_regimen[{plan} vs natural_prices]"]
    revenue = truth[f"ate_revenue[{plan} vs natural_prices]"]
    print(f"  {plan} against natural_prices: renewal {renewal:+.4f}  revenue {revenue:+.4f}")

rows and columns: (10000, 9)
['usage', 'engaged_1', 'price_1', 'engaged_2', 'price_2', 'engaged_3', 'price_3', 'renewed', 'revenue']
   usage  engaged_1  price_1  engaged_2  price_2  engaged_3  price_3  renewed  revenue
0  0.053        0.0    6.586        1.0    9.119        1.0    7.696      1.0    7.696
1  0.260        0.0    4.130        0.0    6.248        1.0    6.872      1.0    6.872
2 -0.072        1.0    8.736        0.0    6.877        1.0    7.356      1.0    7.356
3 -0.817        1.0    7.768        1.0    8.664        1.0    8.672      1.0    8.672
4  0.071        1.0    7.596        0.0    4.978        0.0    7.147      0.0    0.000
5 -0.860        0.0    6.818        0.0    4.364        1.0    6.299      0.0    0.000

missing values: 0
known values of the synthetic law:
  natural_prices  mean renewal 0.7301  mean revenue 5.3765
  price_up        mean renewal 0.6298  mean revenue 5.0643
  price_down      mean renewal 0.8450  mean revenue 5.5019
  price_up against natural_

**What this output tells you.** The frame has 10000 rows and 9 columns. It has no missing value,
because these data have no censoring.

| column | role |
| --- | --- |
| `usage` | a baseline covariate, a standard-normal draw cut off at `2.5` in size |
| `engaged_1`, `engaged_2`, `engaged_3` | the engagement score of that month, recorded before the price |
| `price_1`, `price_2`, `price_3` | the renewal price of that month. The generator does not round it |
| `renewed` | 1 if the subscriber renews at the month-3 price |
| `revenue` | `renewed` times `price_3` |

The true values come from `pricing_truth()`, which integrates the law by quadrature and does not
simulate it.

Under the engine's own prices, the renewal share is 0.7301 and the mean revenue is
5.3765. Under `price_up` they are 0.6298 and 5.0643. Under `price_down` they are 0.8450 and 5.5019.
The true renewal contrast is -0.1003 for `price_up` and 0.1149 for `price_down`. The true revenue
contrast is -0.3122 for `price_up` and 0.1254 for `price_down`.

A real program has no `truth`. Every comparison against it below is a teaching device.

## Step 3: association first

A confounder is a variable that changes both the next price and the outcome. Engagement is one. It
moves the band and the centre of the price, and it raises the renewal. A price also changes the
next engagement. The code reads three arrows of the graph in the data. These tables show
association only.

In [3]:
gap_1 = frame["price_1"] - (6 + 2 * frame["engaged_1"])
print("arrow A_1 -> L_2, share engaged in month 2, by the month-1 price against its centre:")
print(
    frame.groupby(np.where(gap_1 > 0, "above the centre", "at or below"))["engaged_2"]
    .mean()
    .round(3)
    .to_string()
)
print()
print("arrow L_t -> A_t, mean price by that month's engagement:")
by_engagement = {
    f"month {t}": frame.groupby(f"engaged_{t}")[f"price_{t}"].mean() for t in (1, 2, 3)
}
print(pd.DataFrame(by_engagement).round(3).to_string())
print()
print("arrow L_3 -> Y, share renewed, by the month-3 engagement:")
print(frame.groupby("engaged_3")["renewed"].mean().round(3).to_string())

arrow A_1 -> L_2, share engaged in month 2, by the month-1 price against its centre:
above the centre    0.619
at or below         0.696

arrow L_t -> A_t, mean price by that month's engagement:
     month 1  month 2  month 3
0.0    5.946    5.973    5.948
1.0    8.033    8.006    7.980

arrow L_3 -> Y, share renewed, by the month-3 engagement:
engaged_3
0.0    0.598
1.0    0.809


**What this output tells you.** Each table confirms one arrow of the graph.

| arrow | what the table shows |
| --- | --- |
| `A_1` to `L_2` | after a month-1 price above its centre, a share of 0.619 of the subscribers are engaged in month 2. After a price at or below the centre the share is 0.696 |
| `L_t` to `A_t` | the mean month-1 price is 5.946 for subscribers who are not engaged and 8.033 for engaged subscribers. Months 2 and 3 show the same gap of about two |
| `L_3` to `Y` | the renewal share is 0.598 for subscribers who are not engaged in month 3 and 0.809 for engaged subscribers |

Engagement is therefore a confounder of every later price, and an earlier price changes it. One
outcome regression cannot condition on it without blocking part of the earlier price effect. It
cannot leave it out without confounding the later price.

## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs.
[Point-treatment TMLE](point-treatment-tmle.ipynb#step-4-write-the-protocol) introduces its fields
and its fingerprint. The helper `pricing_protocol()` returns the record for this program. It names
the three strategies of the question, and it states that the data have no censoring.

In [4]:
from cleverly.datasets import pricing_protocol

protocol = pricing_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 93155a20f93c3c25
target population: Active monthly subscribers at the start of the pricing window
eligibility: ['Active subscription at the first monthly renewal price', 'A renewal price set by the pricing engine in each of months 1 to 3', 'One account per customer, so shared-household accounts are excluded', 'A subscription that stays open through the month-3 renewal, so a cancellation-free window is a condition of the target population']
time zero: The first monthly renewal price, after the usage score and the month-1 engagement score and before the engine sets the price
treatment strategies: ["The pricing engine's own price in months 1 to 3", "Raise each monthly price by a declared factor where the raised price stays inside that month's price band, otherwise keep the engine's price. The question uses a 12% rise, and the support step also reads 6% and 25%", "Cut each monthly price by 15% where the cut price stays inside that month's price band, otherw

**What this output tells you.** The first line gives the schema version and the fingerprint
`93155a20f93c3c25`. The other lines repeat each field. Read them as a checklist.

| protocol field | the question it answers for this program |
| --- | --- |
| target population and eligibility | who the effect is about. Eligibility includes a subscription that stays open through the month-3 renewal, so a cancellation-free window is a condition of the population |
| time zero | the first monthly renewal price, after the usage score and the month-1 engagement score and before the engine sets the price |
| treatment strategies and versions | the engine's own prices, a price rise by a declared factor, and a 15% cut. The question uses a 12% rise, and Step 9 also reads 6% and 25%. Each plan quotes the new price for one month |
| outcome and horizon | the renewal at the month-3 price, and the month-3 revenue |
| intercurrent-event handling | cancellation before the month-3 renewal is outside the population, so every subscriber has all three prices and the renewal. The data have no censoring |
| interference unit | the individual customer |
| assumption rationale | the engine log supports strong sequential randomization given the usage and the monthly engagement. The declared band supports positivity. One account per customer supports no interference |

## Step 5: design and identification

`LongitudinalTreatment` reads the wide frame. The argument `treatment` lists the price columns in
time order. The argument `time_varying` lists the columns recorded before each price, one list for
each month. The argument `baseline` lists `usage`. The argument `continuous_treatment` marks each
price as a continuous node and declares its support. `PRICE_BANDS` maps each price column to the
band $[4 + 2e, 8 + 2e]$ for the engagement $e$ of that month.

A plan is a `DynamicRegimen` with one policy for each month. A policy is a `ModifiedPolicy` of
pieces. On each interval of the price, a piece applies one increasing map with a known inverse. The
declaration `policy_kind="known"` says that the policy is a fixed function, chosen before the fit.
The dictionary `PRICE_PLANS` holds the three plans. The typed `RegimeContrast` compares each plan
with `natural_prices`.

In [5]:
from cleverly import CausalStudy, LongitudinalTreatment, RegimeContrast
from cleverly.datasets import PRICE_BANDS, PRICE_PLANS


def pricing_design(outcome):
    return LongitudinalTreatment(
        outcome=outcome,
        treatment=["price_1", "price_2", "price_3"],
        baseline=["usage"],
        time_varying=[["engaged_1"], ["engaged_2"], ["engaged_3"]],
        continuous_treatment=PRICE_BANDS,
    )


study = CausalStudy(frame, design=pricing_design("renewed"), protocol=protocol)
for month in (1, 2, 3):
    print(f"history_names({month}):", study.data.history_names(month))
print()
print("plans:", list(PRICE_PLANS))
effect = study.identify(RegimeContrast(PRICE_PLANS, reference="natural_prices"))
print(effect.summary(protocol="fingerprint"))

history_names(1): ('usage', 'engaged_1')
history_names(2): ('usage', 'engaged_1', 'engaged_2')
history_names(3): ('usage', 'engaged_1', 'engaged_2', 'engaged_3')

plans: ['natural_prices', 'price_up', 'price_down']
contrast of each regime against the reference
identified by explicit-adjustment: sequential g-formula under the declared treatment regimen
history at price_1: ['usage', 'engaged_1']
history at price_2: ['usage', 'engaged_1', 'engaged_2', 'price_1']
history at price_3: ['usage', 'engaged_1', 'engaged_2', 'engaged_3', 'price_1', 'price_2']
required nuisances: ['sequential_outcome_regressions', 'treatment_and_censoring_mechanisms']
assumptions:
  - consistency: each observed history equals the potential history under its realized regimen
  - no interference: one unit's potential history does not depend on other units' regimens
  - sequential exchangeability in the form the chosen interpretation names: strong sequential randomization (Díaz et al. 2023, Assumption 3) for the modi

**What this output tells you.** The three `history_names` lines list the columns that the design
puts in the history of each month. Month 1 holds `usage` and `engaged_1`, the confounder of the
first price. Each later month adds that month's engagement. The earlier prices join the history
through the policy, and the `history at` lines of the summary show them.

The `history at price_3` line lists `engaged_1`, `engaged_2`, `engaged_3`, `price_1`, and `price_2`,
with `usage`. The summary names the estimand and its identification, the sequential g-formula under
the declared treatment regimen. The last line prints the fingerprint from Step 4.

The summary lists two interpretations of the policy. The first is the modified treatment policy
effect: the mean outcome had each month's natural price been replaced by the policy's price, with the
history that the earlier replacements produce. The question of this page asks for it. It needs
strong sequential randomization. The second is a stochastic intervention that draws the price from
the law of the policy's price. It needs only standard sequential randomization, and it answers a
different question.

The printed assumptions are those of the table in the plan. Strong sequential randomization is
scientific judgment, and Step 11 says what could break it. Step 9 reads the empirical part of
positivity.

## Step 6: estimate the renewal contrasts with LTMLE

The configuration is written out in full.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | logistic regression | fits the regression of the binary renewal at the last node |
| `pseudo_learner` | linear regression | fits the earlier regressions. Their targets are continuous, and the recursion clips their predictions to the unit interval |
| `treatment_learner` | degree-2 logistic classifier | estimates each density ratio by classification. It takes the degree-2 terms of the history and the price, scales them, and fits a logistic regression with a large penalty constant. It has no indicator for the pieces of a plan |
| `ratio` | `"classifier"` | the route that the registered study of this law uses. A learned density is the other route |
| `CrossFitting(enabled=False)` | in-sample fit | the models have few parameters, and the registered cells for this configuration fit in sample |
| `g_bounds` | the default `(0.01, 1)` | bounds the censoring and categorical treatment factors. This fit has neither, so it bounds nothing. Step 9 reads the weights directly |
| `Runtime(random_state=1000, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

The code then compares each contrast with its true value. It prints the `support` label of each
estimate and the reason that the label gives.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

logistic = LogisticRegression(max_iter=1000, random_state=42)
ratio_model = make_pipeline(
    PolynomialFeatures(2, include_bias=False),
    StandardScaler(),
    LogisticRegression(C=1e6, max_iter=2000, random_state=42),
)
ltmle = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic,
        pseudo_learner=LinearRegression(),
        treatment_learner=ratio_model,
        ratio="classifier",
    ),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=1000, n_jobs=1),
)
result = effect.estimate(method=ltmle)
print(result.summary(protocol="fingerprint"))


def against_truth(fit, truth_kind):
    rows = []
    for key, estimate in fit.estimates.items():
        plan = key.removeprefix("ate_regimen[").removesuffix(" vs natural_prices]")
        exact = truth[f"{truth_kind}[{plan} vs natural_prices]"]
        low, high = estimate.ci
        rows.append(
            {
                "contrast": plan,
                "estimate": estimate.psi,
                "std_error": estimate.std_error,
                "ci_low": low,
                "ci_high": high,
                "truth": exact,
                "covers": bool(low <= exact <= high),
                "support": estimate.support.label,
            }
        )
    return pd.DataFrame(rows).set_index("contrast")


print()
print(against_truth(result, "ate_regimen").round(4).to_string())
print()
for key, estimate in result.estimates.items():
    print(f"{key}: {estimate.support.label}. {estimate.support.reason}")

Longitudinal TMLE (3 time points, n = 10000)

parameter                                  estimate  std. error  95% CI              p-value
-----------------------------------------  --------  ----------  ------------------  -------
ate_regimen[price_up vs natural_prices]    -0.1026   0.0065      [-0.1152, -0.0899]  <1e-4  
ate_regimen[price_down vs natural_prices]  0.1050    0.0079      [0.0896, 0.1205]    <1e-4  

  time points: 3
  outcome family: binomial
  regimens: natural_prices=(m:natural course/m:natural course/m:natural course), price_up=(m:price_up at 1/m:price_up at 2/m:price_up at 3), price_down=(m:price_down at 1/m:price_down at 2/m:price_down at 3)
    a 'd' is a rule d_t(H_t) read off [W, L_1, ..., L_t]; its followers are a covariate-dependent set, so the counts below describe this sample
    assigned arms, natural_prices: c51a991afd938ee0
    assigned arms, price_up: ed3314cb33273f4d
    assigned arms, price_down: 517e7cb67a19cdff
    a 'q' is a known policy q_t(a | H_t

**What this output tells you.** The table reports two contrasts against `natural_prices`. The
`price_up` contrast is -0.1026, with a standard error of 0.0065 and a 95% interval of
(-0.1152, -0.0899). A 12% rise lowers the month-3 renewal share by about ten points. The interval
contains the true contrast of -0.1003.

The `price_down` contrast is 0.1050, with an interval of (0.0896, 0.1205). It contains the true
0.1149. Both intervals contain their true values on this draw. That is one draw, not a coverage
result.

The review probe `reviews/notebook-review/probes/sequential-pricing/sweep.py` repeats the fits of
this page on 200 draws, seeds 1000 to 1199. The two intervals covered on 186 and 187 of 200 draws.
The bias of the estimates was -0.0002 and 0.0004, each with a Monte Carlo standard error of 0.0005.

The `support` label of each estimate is `supported`, and the reason names the registered row.

The label means that a registered row covers this configuration: the engine, the number of nodes, the
splitting, the outcome scale, and the ratio route. It does not mean that a study measured this
learner, this plan, or these factors. The [support
label](../user-guide/results-assessment.md#read-the-support-label) makes no coverage claim. The
method-evidence section lists the gaps. The summary also prints simultaneous bands. They carry the
label `experimental`, and this page does not use them.

## Step 7: estimate the revenue contrasts

Revenue is `renewed` times the month-3 price. It is a continuous outcome with the declared support
$[0, 10]$. A fit of a modified treatment policy refuses a continuous outcome without `q_bounds`, so
the code declares `Targeting(q_bounds=(0.0, 10.0))`. The fit never clips an outcome. See the
[limits of the continuous-policy
evidence](../technical-reference/longitudinal-tmle.md#limits-of-the-continuous-policy-evidence).

The outcome learner is a linear regression. It is cheap, and it does not match the law, because
revenue is zero for every subscriber who lapses and the price for every subscriber who renews.
Everything else stays as in Step 6.

In [7]:
from cleverly import Targeting

revenue_method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(),
        pseudo_learner=LinearRegression(),
        treatment_learner=ratio_model,
        ratio="classifier",
    ),
    targeting=Targeting(q_bounds=(0.0, 10.0)),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=1000, n_jobs=1),
)
revenue_study = CausalStudy(frame, design=pricing_design("revenue"), protocol=protocol)
revenue_result = revenue_study.identify(
    RegimeContrast(PRICE_PLANS, reference="natural_prices")
).estimate(method=revenue_method)
print(against_truth(revenue_result, "ate_revenue").round(4).to_string())
print()
for key, estimate in revenue_result.estimates.items():
    print(f"{key}: {estimate.support.label}. {estimate.support.reason}")

            estimate  std_error  ci_low  ci_high   truth  covers    support
contrast                                                                   
price_up      -0.313     0.0557 -0.4223  -0.2038 -0.3122    True  supported
price_down     0.074     0.0539 -0.0316   0.1796  0.1254    True  supported

ate_regimen[price_up vs natural_prices]: supported. Row G39.ltmle.tau3_revenue covers this configuration.
ate_regimen[price_down vs natural_prices]: supported. Row G39.ltmle.tau3_revenue covers this configuration.


**What this output tells you.** The `price_up` revenue contrast is -0.3130, with an interval of
(-0.4223, -0.2038). The true contrast is -0.3122. A 12% rise loses revenue, because the subscribers
who lapse outweigh the higher price of those who stay.

The `price_down` revenue contrast is 0.0740, with an interval of (-0.0316, 0.1796). The true contrast
is 0.1254. The interval contains zero on this draw, so this fit does not tell whether the cut raises
revenue. In the review probe, the interval of the cut contained zero on 60 of 200 draws.

The linear outcome regression is wrong for revenue, and the probe shows a small bias. Over 200 draws
the estimates of the `price_up` and `price_down` contrasts had biases of 0.0092 and 0.0083, each
about two Monte Carlo standard errors (0.0040 and 0.0038). The standard deviations of the estimates
are 0.0559 and 0.0538, so each bias is about a sixth of them. The intervals covered on 185 and 191 of 200 draws. The
sweep raised `SparseInfluenceWarning` on 55 of 200 revenue fits. This draw raised none.

The probe `revenue_learner.py` refits 60 draws with a quadratic outcome learner. For the `price_up`
contrast the bias was 0.0028 with the quadratic learner and 0.0126 with the linear learner, each
with a Monte Carlo standard error of 0.0075. The quadratic learner lowers the bias estimate, and 60
draws cannot separate the two. The probe does not show that the linear learner carries the bias.

Both revenue fits are `supported`, and the reason names a registered row for this configuration. The
label does not cover the linear outcome learner or a revenue contrast. The method-evidence section
says so.

## Step 8: the failure mode, one regression for a sequential question

The question asks for a policy effect. A regression gives an association. The natural tool is one
logistic regression of `renewed` on the usage, the three engagement scores, and the three prices,
with the prices replaced by the plan's prices. This regression is correctly specified: the true
logit is linear in exactly these columns. It still answers a different question.

| for the comparison | on this page |
| --- | --- |
| estimands | LTMLE targets the policy effect: the mean renewal when the later engagement and the engine's later prices respond to the plan's prices. The regression with engagement targets the mean renewal under the plan's prices, with engagement and the engine's next natural prices held at the values they took under the engine's own prices. That is a natural-direct-effect type of quantity. It is a different estimand. The regression without engagement targets an association |
| assumptions added | the regression adds that its model is correct, and it adds no sequential argument. A correct model does not repair the estimand |
| mechanism | a price changes the next engagement ($A_t \to L_{t+1} \to Y$) and the engine's next price ($A_t \to A_{t+1} \to Y$). The regression holds both at their recorded values, which blocks both paths. Without engagement, $L_t$ raises both the price and the renewal, so a high price looks harmless. Hernán and Robins (Chapter 20) call the first case treatment-confounder feedback |
| evidence | `pricing_truth()` gives the exact limit of each analysis. The code prints them beside the fits |
| reading | one draw shows no general property. The review probe repeats the comparison |

The two regressions are scikit-learn code and not `cleverly` fits, so the code runs them inside
`with thread_limit():`. The function `planned_prices` writes each plan's rule. It multiplies the
engine's price by the factor where the moved price stays inside the band, and it keeps the price
elsewhere.

In [8]:
from cleverly.learners import thread_limit


def planned_prices(plan, month):
    price = frame[f"price_{month}"].to_numpy()
    engaged = frame[f"engaged_{month}"].to_numpy()
    if plan == "price_up":
        return np.where(price < (8 + 2 * engaged) / 1.12, 1.12 * price, price)
    return np.where(price < (4 + 2 * engaged) / 0.85, price, 0.85 * price)


def one_regression(columns, plan):
    with thread_limit():
        model = LogisticRegression(max_iter=2000, C=1e6).fit(frame[columns], frame["renewed"])
        moved = frame[columns].copy()
        for month in (1, 2, 3):
            moved[f"price_{month}"] = planned_prices(plan, month)
        return float(
            np.mean(model.predict_proba(moved)[:, 1] - model.predict_proba(frame[columns])[:, 1])
        )


prices = ["price_1", "price_2", "price_3"]
adjust_columns = ["usage", "engaged_1", "engaged_2", "engaged_3", *prices]
omit_columns = ["usage", *prices]
rows = []
for plan in ("price_up", "price_down"):
    key = f"ate_regimen[{plan} vs natural_prices]"
    low, high = result[key].ci
    rows.append(
        {
            "plan": plan,
            "truth": truth[key],
            "ltmle": result[key].psi,
            "ltmle_low": low,
            "ltmle_high": high,
            "adjust": one_regression(adjust_columns, plan),
            "omit": one_regression(omit_columns, plan),
        }
    )
print(pd.DataFrame(rows).set_index("plan").round(4).to_string())
print()
print("exact effect when the plan's prices enter the renewal model and a path stays natural:")
paths = {
    plan: {
        "truth": truth[f"ate_regimen[{plan} vs natural_prices]"],
        "engagement held": truth[f"ate_engagement_held[{plan}]"],
        "engine held": truth[f"ate_engine_held[{plan}]"],
        "both held": truth[f"ate_both_held[{plan}]"],
    }
    for plan in ("price_up", "price_down")
}
print(pd.DataFrame(paths).round(4).to_string())

             truth   ltmle  ltmle_low  ltmle_high  adjust    omit
plan                                                             
price_up   -0.1003 -0.1026    -0.1152     -0.0899 -0.0796 -0.0202
price_down  0.1149  0.1050     0.0896      0.1205  0.0938  0.0268

exact effect when the plan's prices enter the renewal model and a path stays natural:
                 price_up  price_down
truth             -0.1003      0.1149
engagement held   -0.0906      0.1050
engine held       -0.0900      0.1081
both held         -0.0804      0.0973


**What this output tells you.** The first table gives, for each plan, the true contrast, the LTMLE
estimate with its interval, and the two regressions. The `adjust` column is the regression with
engagement. The `omit` column is the regression without it.

For `price_up`, the truth is -0.1003, and LTMLE gives -0.1026. The `adjust` regression gives
-0.0796, which lies outside the LTMLE interval (-0.1152, -0.0899) on this draw. The `omit`
regression gives -0.0202.

The second table gives the exact limits. The `adjust` regression estimates the `both held` value,
-0.0804 for `price_up`, and not the truth. Each path supplies about half of the gap. Holding
engagement at its natural values gives -0.0906. Holding the engine's next price gives -0.0900.
The regression is correctly specified, so the gap is structural and not a fitting error.

For `price_down`, the truth is 0.1149, and LTMLE gives 0.1050 with an interval of (0.0896, 0.1205).
The `adjust` regression gives 0.0938, and it lies inside that interval on this draw. Its exact limit
is 0.0973, which is further from the truth than the LTMLE estimate is. The `omit` regression gives
0.0268.

The `omit` regression targets an association. Engaged subscribers get higher prices and renew more
often, so a high price looks harmless and the estimate shrinks toward zero. Its limit is an
approximation from one draw of 400000 customers: -0.0203 for `price_up` and 0.0270 for
`price_down`. See the probe `summarize.py`.

The review probe repeats the comparison on 200 draws. The `adjust` regression was further from the
truth than LTMLE on 193 of 200 draws for `price_up` and on 184 for `price_down`. It lay outside the
LTMLE interval on 193 draws for `price_up` and on 127 for `price_down`. The `omit` regression lay
outside the LTMLE interval on every draw for both plans. The `adjust` estimate lay between the LTMLE
estimate and the `omit` estimate on 200 and 199 of 200 draws.

Report the LTMLE estimate. The regressions answer other questions, and a correct outcome model does
not repair that.

## Validation: what each check can and cannot show

Steps 9 to 11 check the fit. None of them checks an identification assumption. Each reading names
what its output does not show.

## Step 9: support, a larger price move rests on fewer customers

A larger factor pushes more natural prices past $u_t / \text{factor}$. The plan then leaves more
prices at the engine's value. The moved prices carry a larger density ratio, and they concentrate
that weight on fewer subscribers. The effect of the move also saturates.

The code fits a 6% rise, the 12% rise of the question, and a 25% rise against `natural_prices` in
one LTMLE fit. The function `price_rise(factor)` builds each plan. The code prints the held share at
month 3 and the estimate for each plan. It then prints the support of month 3: the weights and the
effective sample size.

In [9]:
from cleverly.datasets import price_rise

rises = {
    "natural_prices": PRICE_PLANS["natural_prices"],
    "up_6": price_rise(1.06),
    "price_up": PRICE_PLANS["price_up"],
    "up_25": price_rise(1.25),
}
rise_fit = study.identify(RegimeContrast(rises, reference="natural_prices")).estimate(method=ltmle)
support = rise_fit.diagnostics.support().to_frame()
month3 = support[(support["time"] == 3) & (support["regimen"] != "natural_prices")]
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print(month3[columns].round(3).to_string(index=False))
print()
factors = {"up_6": 1.06, "price_up": 1.12, "up_25": 1.25}
ceiling = 8 + 2 * frame["engaged_3"]
rows = []
for plan, factor in factors.items():
    key = f"ate_regimen[{plan} vs natural_prices]"
    low, high = rise_fit[key].ci
    rows.append(
        {
            "plan": plan,
            "held_share": float((frame["price_3"] >= ceiling / factor).mean()),
            "estimate": rise_fit[key].psi,
            "truth": truth[key],
            "ci_width": high - low,
        }
    )
print(pd.DataFrame(rows).set_index("plan").round(4).to_string())
saturation = (
    truth["ate_regimen[up_25 vs natural_prices]"] / truth["ate_regimen[up_6 vs natural_prices]"]
)
print(f"exact effect of up_25 over up_6: {saturation:.2f}; a linear response gives {25 / 6:.2f}")
print()
for key, estimate in rise_fit.estimates.items():
    print(f"{key}: {estimate.support.label}")

 regimen  time  n_followed  max_weight  effective_n  share_truncated
    up_6     3       10000       5.744     6576.777              0.0
price_up     3       10000      13.697     3516.758              0.0
   up_25     3       10000      19.686     1995.852              0.0

          held_share  estimate   truth  ci_width
plan                                            
up_6          0.0862   -0.0599 -0.0587    0.0131
price_up      0.1916   -0.1026 -0.1003    0.0253
up_25         0.4505   -0.1223 -0.1283    0.0389
exact effect of up_25 over up_6: 2.19; a linear response gives 4.17

ate_regimen[up_6 vs natural_prices]: supported
ate_regimen[price_up vs natural_prices]: supported
ate_regimen[up_25 vs natural_prices]: supported


**What this output tells you.** The first table lists the weights of month 3. The second table lists
the held share and the estimate of each plan.

| plan | held share at month 3 | max weight | effective n of 10000 | width of the 95% interval |
| --- | --- | --- | --- | --- |
| `up_6` | 0.0862 | 5.744 | 6576.777 | 0.0131 |
| `price_up` | 0.1916 | 13.697 | 3516.758 | 0.0253 |
| `up_25` | 0.4505 | 19.686 | 1995.852 | 0.0389 |

A larger factor leaves more prices at the engine's value. The maximum weight rises, the effective
sample size falls, and the interval widens. The estimates are -0.0599, -0.1026, and -0.1223, against
the true -0.0587, -0.1003, and -0.1283. The exact effect saturates. The 25% rise moves the renewal
2.19 times as far as the 6% rise, and a linear response would move it 4.17 times as far.

The weights come from one model: the cumulative density ratio of the degree-2 classifier of Step 6.
The support table describes that model. It cannot see an error in the ratio, and the classifier has
no indicator for the pieces of a plan.

The column `share_truncated` is 0.0 in every row. The argument `g_bounds` bounds only the censoring
and the categorical treatment factors. On complete data it bounds nothing for the price ratio of a
continuous node. A trim of that ratio is open work: [X33, part (d), ratio
trim](../roadmap.md#x33-longitudinal-fitting-options).

A plan that assigned a price outside the declared band would stop the fit. The theory needs the
assigned price inside the support of the engine's price given the history (Assumption 1 of Díaz and
co-authors, 2023). The held share is how the plans of this page stay inside the band.

All three estimates are `supported`. The label covers the configuration and not the factor. No
registered cell measures the 6% and 25% factors.

## Step 10: diagnostics, what the fit can check

`result.assess()` runs the checks that apply to the fit. The code prints the operations that ran. It
then prints the support report of the renewal fit for the three plans of the question. The report
shows the weights at each month. It ends with the nuisance models that the report scores, and the
role that it leaves out.

In [10]:
assessment = result.assess()
ledger = assessment.to_frame()
ran = ledger[~ledger["status"].isin(["unavailable", "not_applicable", "deferred"])]
print(ran[["surface", "check", "status"]].to_string(index=False))
support_report = assessment.report("support").to_frame()
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print()
print(support_report[columns].round(3).to_string(index=False))
kish_ratio = (support_report["effective_n"] / support_report["n_followed"]).min()
print(f"minimum effective-sample-size ratio: {kish_ratio:.1%}")
held_down = (planned_prices("price_down", 3) == frame["price_3"]).mean()
print(f"price_down, share of month-3 prices left at the engine's value: {held_down:.1%}")
print()
nuisance = assessment.report("nuisance_models")
outcome = nuisance.to_frame()
outcome = outcome[(outcome["role"] == "outcome") & (outcome["regimen"] == "price_down")]
print(
    outcome[["role", "time", "n", "evaluation", "auc", "calibration_slope"]]
    .round(3)
    .to_string(index=False)
)
print(
    "not scored:",
    sorted({omission.reason for omission in nuisance.omissions if omission.role == "treatment"}),
)

   surface           check    status
validation score_equations    passed
validation         support completed
validation nuisance_models completed

       regimen  time  n_followed  max_weight  effective_n  share_truncated
natural_prices     1       10000       1.000    10000.000              0.0
natural_prices     2       10000       1.000    10000.000              0.0
natural_prices     3       10000       1.000    10000.000              0.0
      price_up     1       10000       3.352     6939.831              0.0
      price_up     2       10000       7.306     5030.565              0.0
      price_up     3       10000      13.697     3516.758              0.0
    price_down     1       10000       4.643     5206.547              0.0
    price_down     2       10000      13.043     2843.960              0.0
    price_down     3       10000      31.464     1540.808              0.0
minimum effective-sample-size ratio: 15.4%
price_down, share of month-3 prices left at the engine's v

**What this output tells you.** The first table lists the operations that ran. The score-equation
check is `passed`. The support report and the nuisance report are `completed`. A completed result
means the calculation ran. It is not a pass.

The support table divides by the density ratios of the degree-2 classifier. It describes that model.
It cannot see an error in the ratio.

| plan | max weight at month 3 | effective n | what it shows |
| --- | --- | --- | --- |
| `natural_prices` | 1.000 | 10000.000 | equal weights |
| `price_up` | 13.697 | 3516.758 | the weights are uneven, and about a third of the sample carries them |
| `price_down` | 31.464 | 1540.808 | the heaviest weights of the three plans |

The `price_down` plan sends the prices below $l_t / 0.85$ unchanged and moves the rest down. It
leaves 19.7% of the month-3 prices at the engine's value. Its cumulative density ratio is a product
of three monthly ratios. The maximum weight compounds from 4.643 at month 1 to 13.043 at month 2 and
31.464 at month 3. The minimum ratio of effective sample size to
followers is 15.4%, and it belongs to `price_down` at month 3. No weight is truncated.

The nuisance report scores the outcome regression. Its calibration slope is 1.004, and its `auc` is
0.697. The regression is fitted in sample, so a slope near 1 does not show that the model is correct.
The report leaves the density-ratio classifier unscored. Nothing in this report checks that
classifier.

In the review probe, the maximum weight of `price_up` at month 3 averaged 15.07, with a standard
deviation of 2.12. For `price_down` it averaged 32.44, with a standard deviation of 5.40. The
effective sample size over n averaged 0.359 for `price_up` and 0.152 for `price_down`.

## Step 11: strong sequential randomization, what this page cannot probe

The technical reference derives no omitted-variable bound for a regime, shift, or modified
treatment policy contrast ([X9](../roadmap.md#x9-omitted-variable-bounds-on-the-other-linear-functionals)).
So this page runs no sensitivity analysis.

A covariate-drop benchmark cannot run either. The only baseline covariate is `usage`, and `engaged_t`
defines the price band and the pieces of each policy. Dropping either one changes the question.

The assumption at risk is strong sequential randomization. It fails when the engine reads an input
that the log does not record. An example is a competitor's price or a manual override by a pricing
analyst. Such an input moves the price and the renewal together, and no output on this page would
show it. The engine log is the argument for the assumption. It is not a test.

## Method evidence

The fits on this page use one draw of a synthetic law. The registered study in the table measures the
method over repeated draws. The study page gives its law, its nuisance models, and the result of each
cell.

| fit on this page | registered study | law of the study |
| --- | --- | --- |
| Step 6, LTMLE of the renewal contrasts | [MTP pricing](../technical-reference/method-evidence/mtp-pricing.md) | this page's law |
| Step 7, LTMLE of the revenue contrasts | [MTP pricing](../technical-reference/method-evidence/mtp-pricing.md) | this page's law |
| Step 9, the factor ladder, for `price_up` only | [MTP pricing](../technical-reference/method-evidence/mtp-pricing.md) | this page's law |

The registered study leaves four parts of this page uncovered.

| part of the page | the gap |
| --- | --- |
| the density ratio | the registered cells read a learned ratio only for the `price_up` mean, and not for a contrast or for `price_down` |
| the ratio classifier | the study's learned ratio is a screened quadratic classifier with indicators for the pieces of the plan. The degree-2 classifier of this page has none |
| revenue | one cell measures the `price_up` mean with a correct outcome model. No cell measures a revenue contrast, `price_down`, or a linear outcome learner |
| the factors | no cell measures the 6% and 25% factors |

A review probe is a script under `reviews/notebook-review/probes/`. The step readings quote its
results. A review probe is not a registered study. The probe of this page repeats its configuration
over 200 draws.

The [technical entry](../technical-reference/longitudinal-tmle.md#validation-issues-special-to-this-method)
lists the evidence and the limits of the method. The [limits of the continuous-policy
evidence](../technical-reference/longitudinal-tmle.md#limits-of-the-continuous-policy-evidence)
state the limits that bear on the revenue outcome and the ratio learner.

## What the analysis answers

The question was what share of subscribers would renew in month 3, and what month-3 revenue would
be, under one policy. The policy raises each monthly price by 12% where the raised price stays
inside the band. It keeps the engine's price elsewhere.

| part of the answer | what the page supports |
| --- | --- |
| the estimate | if the assumptions below hold, the 12% rise moves the month-3 renewal share by -0.1026 against the engine's own prices, with a 95% interval of (-0.1152, -0.0899). It moves the mean month-3 revenue by -0.3130, with an interval of (-0.4223, -0.2038). The population is active monthly subscribers in a cancellation-free window. The intervention is the plan's prices for three months, with the later engagement and the engine's later prices responding |
| the cut | the 15% cut raises the renewal share by 0.1050, with an interval of (0.0896, 0.1205). Its revenue effect of 0.0740 has an interval of (-0.0316, 0.1796), which contains zero on this draw. The page does not resolve the sign of the revenue effect of the cut |
| the held prices | each plan leaves about a fifth of the prices at the engine's value, because the moved price would leave the band. The estimate describes the plan with those prices held, and not a pure scale |
| evidence behind the reading | the exact truths, the single-regression comparison, the support report, the factor ladder, and the review probe. The registered study covers this configuration and not this learner, plan, or factor |
| assumptions that stay unverified | strong sequential randomization, consistency, and no interference. The sensitivity step runs no analysis |
| conclusions the page does not support | a general price elasticity, an optimal price, the effect of a price outside the declared band, or a claim about applied data. The law is synthetic and no applied data access exists. A statement that the method is unbiased in general. A revenue effect of the cut |

## Where to go next

The [sequential offers tutorial](sequential-offers.ipynb) fits categorical rules in the same
retailer program. It adds censoring, a declared offer mechanism, and a paired comparison of two
policies. The [time-to-event tutorial](longitudinal-survival.ipynb) handles disenrollment and a
competing event. The [longitudinal user guide](../user-guide/longitudinal.md) shows how to read a
panel and how to report the sequential baselines. [Modified treatment policies at a
node](../technical-reference/longitudinal-tmle.md#modified-treatment-policies-at-a-node) states the
estimator. The [examples index](index.md#the-retailer-program) lists every tutorial in both
programs.